# BiLSTM — Vietnamese PII NER

## 1. Setup & Dependencies


In [ ]:
import os

REPO_DIR = "/content/Vietnamese-PII-NER" if os.path.exists("/content") else os.path.abspath("Vietnamese-PII-NER")
SETUP_MARKER = ".colab_deps_installed"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/PhucTang2005/Vietnamese-PII-NER.git "$REPO_DIR"

%cd $REPO_DIR

# Install dependencies once. Colab must restart after numpy/scikit-learn changes to avoid binary ABI issues.
if not os.path.exists(SETUP_MARKER):
    !pip install -r requirements.txt -q
    with open(SETUP_MARKER, "w", encoding="utf-8") as setup_file:
        setup_file.write("done")
    if os.environ.get("COLAB_RELEASE_TAG"):
        print("Dependencies installed. Restarting Colab runtime; run this cell again after restart.")
        os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed; skipping pip install.")


In [ ]:
# Import libraries and configure the runtime device.
import json
import os
import sys
import numpy as np
import torch
from pathlib import Path
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Ensure the notebook can import the local src package from any Colab working directory.
PROJECT_ROOT_CANDIDATES = [
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    "/content/Vietnamese-PII-NER",
]
PROJECT_ROOT = None
for candidate in PROJECT_ROOT_CANDIDATES:
    if os.path.isdir(os.path.join(candidate, "src")):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing src/. Run the setup cell first."
    )

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from src.models import BiLSTMForNER

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

HF_MODEL_REPO = "Phuc2005/pii-bilstm-ner"
TOKENIZED_DATA_DIR = "./data/tokenized_phobert/"
CHECKPOINT_DIR = "./best_model_bilstm/"


## 2. Load Dataset


In [ ]:
# Load the Vietnamese PII dataset from HuggingFace Hub.
dataset = load_dataset("quynong/cs419-data")
print(dataset)
print("\nTrain sample:")
print(dataset['train'][0])


In [ ]:
# Build the BIO label schema from entity labels present in the dataset.
all_labels = set()
for split in dataset:
    for sample in dataset[split]:
        for ent in sample['privacy_mask']:
            all_labels.add(ent['label'])

all_labels = sorted(all_labels)
print(f"Found {len(all_labels)} entity types:")
print(all_labels)

label_list = ["O"]
for lbl in all_labels:
    label_list.append(f"B-{lbl}")
    label_list.append(f"I-{lbl}")

label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for idx, label in enumerate(label_list)}
num_labels = len(label_list)

print(f"\nTotal BIO labels: {num_labels}")
print(f"First 10: {label_list[:10]}")


## 3. Tokenization & Label Alignment


In [ ]:
# Initialize the PhoBERT tokenizer.
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")


In [ ]:
# Align character-level entity spans to token-level BIO labels.
def char_to_token_labels(source_text, privacy_mask, offset_mapping):
    """Convert character-level entity spans to token-level BIO labels."""
    char_labels = ['O'] * len(source_text)
    for ent in privacy_mask:
        start, end, label = ent['start'], ent['end'], ent['label']
        if start >= len(source_text) or end > len(source_text):
            continue
        char_labels[start] = f"B-{label}"
        for char_idx in range(start + 1, end):
            char_labels[char_idx] = f"I-{label}"

    token_labels = []
    for tok_start, tok_end in offset_mapping:
        if tok_start == 0 and tok_end == 0:
            token_labels.append(-100)
            continue

        if tok_start < len(char_labels):
            label = char_labels[tok_start]
        else:
            label = 'O'
        token_labels.append(label2id.get(label, label2id['O']))

    return token_labels

print("Label alignment function loaded.")


In [ ]:
# Build PhoBERT offset mappings manually because the tokenizer does not return offsets.
def build_phobert_offset_mapping(tokens, text, special_tokens):
    """Build token offsets in raw text for PhoBERT subword tokens."""
    offset_mapping = []
    curr_pos = 0
    lower_text = text.lower()

    for token in tokens:
        if token in special_tokens:
            offset_mapping.append((0, 0))
            continue

        clean_token = token.replace("@@", "")

        while curr_pos < len(text) and text[curr_pos].isspace():
            curr_pos += 1

        if not lower_text[curr_pos:].startswith(clean_token.lower()):
            found = lower_text.find(clean_token.lower(), curr_pos)
            if found != -1:
                curr_pos = found
            else:
                offset_mapping.append((0, 0))
                continue

        start = curr_pos
        end = curr_pos + len(clean_token)
        offset_mapping.append((start, end))
        curr_pos = end

    return offset_mapping


# Tokenize examples and align each entity span to the corresponding PhoBERT tokens.
def tokenize_and_align(examples):
    """Tokenize a batch of raw examples and align BIO labels to PhoBERT tokens."""
    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for source_text, privacy_mask in zip(examples['source_text'], examples['privacy_mask']):
        encoding = tokenizer(
            source_text,
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
        )

        tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        offset_mapping = build_phobert_offset_mapping(
            tokens,
            source_text,
            set(tokenizer.all_special_tokens),
        )

        token_labels = char_to_token_labels(source_text, privacy_mask, offset_mapping)

        while len(token_labels) < MAX_LENGTH:
            token_labels.append(-100)
        token_labels = token_labels[:MAX_LENGTH]

        all_input_ids.append(encoding['input_ids'])
        all_attention_mask.append(encoding['attention_mask'])
        all_labels.append(token_labels)

    return {
        'input_ids': all_input_ids,
        'attention_mask': all_attention_mask,
        'labels': all_labels,
    }

print("Tokenization function loaded for PhoBERT raw text.")


In [ ]:
# Tokenize all dataset splits and save the processed dataset inside the repo.
tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=True,
    batch_size=32,
    remove_columns=dataset['train'].column_names,
    desc="Tokenizing PhoBERT",
)

tokenized_dataset.save_to_disk(TOKENIZED_DATA_DIR)

print(tokenized_dataset)
print(f"\nSample input_ids length: {len(tokenized_dataset['train'][0]['input_ids'])}")
print(f"Sample labels length: {len(tokenized_dataset['train'][0]['labels'])}")
print(f"Tokenized dataset saved to: {TOKENIZED_DATA_DIR}")


## 4. Model Definition


In [ ]:
# Initialize the custom model architecture for token classification.
model = BiLSTMForNER(
    vocab_size=tokenizer.vocab_size,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    padding_idx=tokenizer.pad_token_id,
)
model.to(device)
print(f"Model loaded: BiLSTM with {num_labels} labels")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## 5. Training


In [ ]:
# Configure HuggingFace Trainer for custom model training.
data_collator = DefaultDataCollator()

training_args = TrainingArguments(
    output_dir="./bilstm-ner-pii",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=15,
    weight_decay=1e-4,
    warmup_ratio=0.0,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

print("Training arguments configured for BiLSTM.")


In [ ]:
# Compute entity-level NER metrics and sentence-level entity-detection metrics.
def compute_metrics(eval_preds):
    """Compute NER micro metrics and binary sentence-level PII detection metrics."""
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for pred_seq, label_seq in zip(predictions, labels):
        pred_tags = []
        true_tags = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            pred_tags.append(id2label[pred_id])
            true_tags.append(id2label[label_id])
        true_predictions.append(pred_tags)
        true_labels.append(true_tags)

    ner_precision = precision_score(true_labels, true_predictions, average='micro')
    ner_recall = recall_score(true_labels, true_predictions, average='micro')
    ner_f1 = f1_score(true_labels, true_predictions, average='micro')

    cls_true = []
    cls_pred = []

    for pred_seq, label_seq in zip(predictions, labels):
        has_entity_true = any(label_id not in (-100, label2id['O']) for label_id in label_seq)
        has_entity_pred = any(
            pred_id != label2id['O']
            for pred_id, label_id in zip(pred_seq, label_seq)
            if label_id != -100
        )
        cls_true.append(int(has_entity_true))
        cls_pred.append(int(has_entity_pred))

    cls_precision, cls_recall, cls_f1, _ = precision_recall_fscore_support(
        cls_true, cls_pred, average='binary', zero_division=0
    )
    cls_accuracy = accuracy_score(cls_true, cls_pred)

    return {
        "precision": ner_precision,
        "recall": ner_recall,
        "f1": ner_f1,
        "cls_accuracy": cls_accuracy,
        "cls_precision": cls_precision,
        "cls_recall": cls_recall,
        "cls_f1": cls_f1,
    }

print("Metrics function loaded.")


In [ ]:
# Train the model on the tokenized training split and validate each epoch.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer ready. Starting training BiLSTM...")
trainer.train()


## 6. Evaluation


In [ ]:
# Run final evaluation on the validation split.
eval_results = trainer.evaluate()

print("=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
print("\n" + "-" * 40)
print("NER Entity-Level (Micro):")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")
print(f"  F1:        {eval_results['eval_f1']:.4f}")
print("\n" + "-" * 40)
print("Classification (Has Entity):")
print(f"  Accuracy:  {eval_results['eval_cls_accuracy']:.4f}")
print(f"  Precision: {eval_results['eval_cls_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_cls_recall']:.4f}")
print(f"  F1:        {eval_results['eval_cls_f1']:.4f}")
print("=" * 60)


In [ ]:
# Generate a detailed seqeval report per entity type on the validation split.
eval_output = trainer.predict(tokenized_dataset['validation'])
predictions = np.argmax(eval_output.predictions, axis=-1)
labels = eval_output.label_ids

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    pred_tags = []
    true_tags = []
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            continue
        pred_tags.append(id2label[pred_id])
        true_tags.append(id2label[label_id])
    true_predictions.append(pred_tags)
    true_labels.append(true_tags)

report = classification_report(true_labels, true_predictions, digits=4)
print("Detailed NER Classification Report for PhoBERT (per entity type):")
print(report)


## 7. Inference


In [ ]:
# Load model and run inference on sample text.
from huggingface_hub import snapshot_download

# Download model checkpoint from HuggingFace Hub to local cache
local_ckpt = snapshot_download(repo_id=HF_MODEL_REPO)

inference_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_REPO)
inference_model = BiLSTMForNER.from_pretrained(local_ckpt, device=device)
inference_model.eval()

inference_id2label = inference_model.config.id2label


def predict_entities(text):
    """Run BiLSTM NER inference on a single Vietnamese text."""
    encoding = inference_tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors='pt',
    )

    tokens = inference_tokenizer.convert_ids_to_tokens(encoding['input_ids'][0].tolist())
    offset_mapping = build_phobert_offset_mapping(tokens, text, set(inference_tokenizer.all_special_tokens))
    inputs = {key: value.to(device) for key, value in encoding.items()}

    with torch.no_grad():
        outputs = inference_model(**inputs)
    predictions = torch.argmax(outputs['logits'], dim=-1)[0].cpu().tolist()

    entities = []
    current_entity = None

    for pred_id, (start, end) in zip(predictions, offset_mapping):
        if start == 0 and end == 0:
            continue
        label = inference_id2label[pred_id]

        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {'label': label[2:], 'start': start, 'end': end, 'text': text[start:end]}
        elif label.startswith('I-') and current_entity:
            current_entity['end'] = end
            current_entity['text'] = text[current_entity['start']:end]
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    return entities


text = "Xin chào, tôi là Nguyễn Văn An, SĐT 0912345678, email example@gmail.com"
entities = predict_entities(text)

print(f"Text: {text}")
print(f"\nEntities found: {len(entities)}")
for ent in entities:
    print(f"  [{ent['label']}] \"{ent['text']}\" (pos {ent['start']}-{ent['end']})")


## 8. Save & Export


In [ ]:
# Save the fine-tuned custom model and tokenizer to the repo checkpoint directory.
model.save_pretrained(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Model saved to: {CHECKPOINT_DIR}")
